## Workshop Lab 5: Sharing Scripts and Python APIs

Now that you've seen python at work, it is important to know how you can share your code with others. Methods include:
- Direct file shares (e.g. copying .py or .ipynb files, copying code chunks)
- GitHub (download zip files OR use git to pull and clone repos - this could be a whole other workshop!)
- AGOL and Enterprise services

In this notebook we will briefly go over some ways to share your data, scripts, and project files via ArcGIS Online using python.

### Connecting to ArcGIS Online (AGOL) with Python
Python uses arcgis to connect directly to AGOL. The first step is to establish a portal connection. There are many ways to do this, but for convenience in this workshop, we will use your current ArcPro credentials.

Run the code chunk below. It may take 30 seconds or so.

In [ ]:
from arcgis.gis import GIS

# Method A: If students are already signed into ArcGIS Pro on their machine, 
# this grabs that active session automatically (Recommended for labs)
try:
    gis = GIS("pro")
    print(f"Successfully connected as: {gis.users.me.username} via ArcGIS Pro session.")
except Exception as e:
    print("Could not connect via Pro session. Trying explicit login...")
    
    # Method B: Standard username/password fallback if running in Jupyter standalone
    # (Students can swap in their AGOL credentials or use an ArcGIS Online portal URL)
    portal_url = "https://www.arcgis.com"
    username = input("Enter your AGOL Username: ")
    gis = GIS(portal_url, username)
    print(f"Successfully connected to {gis.properties.portalName} as {gis.users.me.username}.")

Note: This works because you are logged into ArcPro! It is a good practice to avoid hard coding logins and other senstive information.

### Now let's grab a dataset from AGOL.

For this practice, we will be grabbing a dataset on mountain lions from AGOL.

In [ ]:
     
from arcgis.gis import GIS
import os
import pandas as pd

# GIS() with no arguments connects to ArcGIS Online anonymously: no account, no sign-in.
# That means we can only see PUBLIC items. To see private or organization-only items, or to
# upload anything, connect as yourself instead, e.g. GIS("pro") as in the connection cell above.
gis = GIS()
print(gis.users.me)   # Prints None: this connection has no user attached, so it is anonymous.
# For comparison, a signed-in connection would print your username here:
# print(GIS("pro").users.me)

'''
In previous scripts we've used # to denote comments. This chunk shows another way to include comments: three apostrophes in a row bracketing text. 
This allows for multi-line comments or instructions.

Every AGOL item has a unique 32-character ID, found in its URL after "id="; 44424e39f5214282a16328aad26a341c is the unique ID for the mountain lion csv, which I've made public. 
For this script we will feed that ID directly into the gis.content.get() statement below.
This is mostly for convenience and workshop flow since we are not all part of the same organization. 
However, if you are sharing data in this way within your organization (you MUST be signed in as in gis = GIS("pro") above), you could use a code chunk like:

gis = GIS("pro")   # signed in, so the search covers your organization's content

results = gis.content.search('title:mountainlions_data', item_type='CSV')
for r in results:
    print(r.title, r.owner, r.id)   # printing the owner shows whose upload each match is

if not results:
    raise ValueError("No matching item found. Check the title, or whether it's shared with you.")
item = results[0]   # results is a list; [0] picks the first match
print(item.title, item.type)

'''
# Try pasting the org search chunk here:


# To try the org search chunk above, make sure you copy that chunk out of the apostrophes and paste it above this comment. Also, you should comment out the next two lines of code:
item = gis.content.get("44424e39f5214282a16328aad26a341c")
print(item.title, item.type)

# Save into the clean data folder (adjust the folder name to match your project)
save_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "04_data", "clean"))
os.makedirs(save_dir, exist_ok=True)

path = item.download(save_path=save_dir)
print("Saved to:", path)

# Confirm it reads
df = pd.read_csv(path)
df.head()

You should now have a new csv in the 'clean' data folder. Try using the XYtoPoint script tool in ArcPro on it!

Note: If you have trouble downloading using the script chunk above, try pasting this url into your browser:
  https://www.arcgis.com/sharing/rest/content/items/44424e39f5214282a16328aad26a341c/data

### Fetching New Data with an API (application programming interface)
The observations on roadrunners, coyotes, and mountain lions we've been looking at came from iNaturalist.org. They have their own API which we can use to grab more data. There are many many APIs that you can connect with for all kinds of data!

Normally, you could install the `pyinaturalist` library in your python environment and use that to grab data from iNaturalist.org. However, for this workshop we are using the `arcgispro-py3` environment. ArcPro tends to deny library installs to prevent environment corruption, so for the example below we will use a work around (part of the power of code is that almost any problem has a workaround!).

Below we will use a standard python library called `requests` to grab observations on black bears. Run the code chunk:

In [ ]:
import requests
import pandas as pd
import arcpy
import os

print("--- STEP 1: Fetching live data from iNaturalist API ---")
url = "https://api.inaturalist.org/v1/observations"
params = {
    "taxon_name": "Ursus americanus",
    "place_id": 9,  # New Mexico 
    "per_page": 200,  # Keep it small for the workshop - this grabs 200 observations
    "quality_grade": "research"
}

response = requests.get(url, params=params)
data = response.json()
observations = data.get('results', [])

if len(observations) > 0:
    parsed_data = []
    for obs in observations:
        location = obs.get('location')
        if location:  # Skip records missing coordinate data
            lat, lon = location.split(',')
            parsed_data.append({
                'id': obs.get('id'),
                'observed_on': obs.get('observed_on'),
                'time_zone': obs.get('time_zone'),
                'latitude': float(lat),
                'longitude': float(lon),
                'scientific_name': obs.get('scientific_name'),
                'common_name': obs.get('species_guess') # or common name if available from API
            })
    
    df_bears = pd.DataFrame(parsed_data)
    
    # Match Notebook 3's datetime parsing and calculated columns
    df_bears['date_dt'] = pd.to_datetime(df_bears['observed_on'], errors='coerce')
    df_bears['obs_year'] = df_bears['date_dt'].dt.year
    
    # Align to Notebook 3 schema order
    columns_to_keep = [
        'id', 
        'date_dt', 
        'time_zone', 
        'latitude', 
        'longitude', 
        'scientific_name', 
        'common_name',
        'obs_year'
    ]
    df_bears = df_bears[columns_to_keep].copy()
    
    print(f"Successfully pulled and structured {len(df_bears)} valid black bear observations matching Notebook 3 schema.")
    
    print("\n--- STEP 2: Saving CSV to working folder ---")
    target_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "04_data", "working"))
    os.makedirs(target_dir, exist_ok=True)
    
    csv_path = os.path.join(target_dir, "smblack_bears_data.csv")
    df_bears.to_csv(csv_path, index=False)
    print(f"CSV saved to: {csv_path}")
    
    print("\n--- STEP 3: Converting CSV to Local Feature Class ---")
    workspace = os.path.abspath(os.path.join(os.getcwd(), "..", "03_pro_project", "IntroToPythonAndArcade.gdb"))
    arcpy.env.workspace = workspace
    arcpy.env.overwriteOutput = True
    out_fc = "smblack_bears_cleaned"
    
    print(f"Target GDB: {workspace}")
    
    arcpy.management.XYTableToPoint(
        in_table=csv_path,
        out_feature_class=out_fc,
        x_field="longitude",
        y_field="latitude",
        coordinate_system=arcpy.SpatialReference(4326) # WGS 1984
    )
    
    print(f"Success! Created feature class '{out_fc}' in geodatabase with matching schema.")
    
else:
    print("No observations returned. Check query parameters.")

### Pushing Files to AGOL
Now let's practice moving a copy of our new smblack_bears_cleaned feature class to AGOL. 

We will use argis to upload the cleaned csv file. This should be possible regardless of publishing privileges.

In [ ]:
from arcgis.gis import GIS
import os

# Connect to your active ArcGIS Pro portal session
gis = GIS("pro")
print(f"Logged in as: {gis.properties.user.username}")

# Path to your working CSV file
csv_path = os.path.abspath(os.path.join(os.getcwd(), "..", "04_data", "working", "smblack_bears_data.csv"))

# Define item metadata for AGOL
item_properties = {
    "title": "Black Bears New Mexico (CSV)",
    "type": "CSV",
    "tags": ["Python", "ArcGIS Pro", "Black Bears", "New Mexico", "CSV"],
    "snippet": "New Mexico black bear CSV data uploaded for workshop."
}

# Upload the CSV to your AGOL content (without publishing it as a service)
csv_item = gis.content.add(item_properties, data=csv_path)

print(f"Success! Uploaded CSV to AGOL:")
print(f"Title: {csv_item.title}")
print(f"Item ID: {csv_item.id}")

This will upload your csv to the AGOL content of your organization (in this case TGIS). Let's upload our PythonIsCool.atbx as well! (note - if you have your scripts linked from notebook 3, you should change them to be embedded before sharing this toolbox).

In [ ]:
from arcgis.gis import GIS
import os
import zipfile

gis = GIS("pro")

atbx_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "03_pro_project"))
atbx_name = "PythonIsCool.atbx"
atbx_path = os.path.join(atbx_dir, atbx_name)

zip_path = os.path.join(atbx_dir, "PythonIsCool.zip")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(atbx_path, arcname=atbx_name)

item_properties = {
    "title": "Python Is Cool Toolbox",
    "type": "Code Sample",
    "tags": ["Python", "ArcGIS Pro", "Toolbox", "ATBX"],
    "snippet": "PythonIsCool toolbox."
}

atbx_item = gis.content.add(item_properties, data=zip_path)

print(f"Uploaded: {atbx_item.title} ({atbx_item.id})")

#### Other things you can share (examples only)

The chunks below show the same pattern (connect, describe the item, add it) applied to other kinds of files. They are **not meant to be run during the workshop**. Some need publisher permissions, and all of them would create new items in your AGOL account, which eats up tokens. They are included as reference for later if you want to try this at home.

*These are patterns, not tested code. Check the arguments against the Esri documentation for your version before using them.*

<details>
<summary><b>Publish a feature class as a hosted layer (needs publisher privileges)</b></summary>

```
from arcgis.gis import GIS
import arcpy
import os

# Connect to your active ArcGIS Pro portal session
gis = GIS("pro")
print(f"Logged in as: {gis.properties.user.username}")

# Define paths
workspace = os.path.abspath(os.path.join(os.getcwd(), "..", "03_pro_project", "IntroToPythonAndArcade.gdb"))
fc_name = "smblack_bears_cleaned"
fc_path = os.path.join(workspace, fc_name)

# Create a temporary output path for GeoJSON
temp_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "04_data", "working"))
geojson_path = os.path.join(temp_dir, "black_bears_temp.geojson")

# Export the geodatabase feature class to GeoJSON using the correct parameter
print(f"Exporting feature class '{fc_name}' to GeoJSON...")
arcpy.conversion.FeaturesToJSON(
    in_features=fc_path,
    out_json_file=geojson_path,
    format_json="NOT_FORMATTED",
    include_z_values=False,
    include_m_values=False,
    geoJSON="GEOJSON"
)

# Define item metadata for AGOL
item_properties = {
    "title": "Black Bears New Mexico (Feature Class)",
    "type": "GeoJson",
    "tags": ["Python", "ArcGIS Pro", "Black Bears", "New Mexico", "Feature Class"],
    "snippet": "New Mexico black bear feature class published from project GDB for workshop."
}

# Add the GeoJSON item to AGOL and publish it as a hosted feature layer
print("Uploading and publishing to ArcGIS Online...")
geojson_item = gis.content.add(item_properties, data=geojson_path)
hosted_feature_layer = geojson_item.publish()

print(f"Success! Published hosted feature layer to AGOL:")
print(f"Title: {hosted_feature_layer.title}")
print(f"Item ID: {hosted_feature_layer.id}")
```
</details>

<details>
<summary><b>Share a project folder as a zip file</b></summary>

```
from arcgis.gis import GIS
import os
import shutil

gis = GIS("pro")

project_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "03_pro_project"))
zip_output = os.path.abspath(os.path.join(os.getcwd(), "..", "03_pro_project_archive"))

print("Zipping project directory...")
shutil.make_archive(zip_output, 'zip', project_dir)
zip_path = zip_output + ".zip"

item_properties = {
    "title": "Intro To Python And Arcade Project",
    "type": "Code Sample",
    "tags": ["Python", "ArcGIS Pro", "APRX", "Project"],
    "snippet": "Archived ArcGIS Pro project files."
}

project_item = gis.content.add(item_properties, data=zip_path)

print(f"Uploaded: {project_item.title} ({project_item.id})")
```
</details>

<details>
<summary><b>Share a project as a project package (.ppkx) using arcpy</b></summary>

```
import arcpy
from arcgis.gis import GIS
import os

gis = GIS("pro")

project_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "03_pro_project"))
ppkx_path = os.path.join(project_dir, "IntroToPythonAndArcade.ppkx")

p = arcpy.mp.ArcGISProject("CURRENT")
arcpy.management.PackageProject(
    in_project=p.filePath,
    output_file=ppkx_path,
    sharing_internal="INTERNAL"
)

item_properties = {
    "title": "Intro To Python And Arcade Project Package",
    "type": "Project Package",
    "tags": ["Python", "ArcGIS Pro", "Project Package", "PPKX"],
    "snippet": "Packaged ArcGIS Pro project."
}

ppkx_item = gis.content.add(item_properties, data=ppkx_path)

print(f"Uploaded: {ppkx_item.title} ({ppkx_item.id})")
```
</details>

Many users don't have publishing privileges (I don't, for example), but uploading csv files or zipped aprx toolboxes should still work!

## Workshop Conclusion: From Raw Code to Enterprise GIS

Congratulations! Over the past two days, you have built a complete, repeatable GIS data pipeline from scratch. Let's look back at what you accomplished:
1. **Notebook 1 & 2:** Jumped into Jupyter notebooks, learned about environments, loaded raw public CSV observations, and visualized points natively.
2. **Notebook 3:** Transitioned from manual table steps to automated data cleaning, scaling up to thousands of records, and turning Python logic into a custom `.atbx` script tool.
3. **Notebook 4:** Used ArcGIS Pro's rendering engine with Arcade for dynamic seasonal symbology, custom labeling, spatial buffer popups, and attribute based scaling—all without altering the underlying data.
4. **Notebook 5:** Bridged desktop GIS to the cloud, utilizing the Python API to query web services, fetch live API data, and package projects for organizational sharing.
5. **Bonus, Notebook 5b:** If you finish early, 05b_Challenge puts the whole pipeline to work on a new species.

You are scripting, automating, and scaling your GIS workflows!

## Continuing Your Learning!
One of the best ways to continue your learning journey is to keep coding. Python has an incredible array of free and structured learning resources to help you along the way. The best place to start is with the *Tribal GIS Online Intro to Python and Intro to Arcade Courses*. Below are some additional recommendations:

### Python for GIS 
* **[Learn ArcGIS (learn.arcgis.com)](https://learn.arcgis.com/):** Free, guided lessons using real-world data specifically for ArcGIS API for Python and ArcGIS Pro workflows.
* **[ArcGIS API for Python Documentation](https://developers.arcgis.com/python/):** The official guide and API reference for automating web GIS, managing users, and working with content in ArcGIS Online.
* **[Python Scripting for ArcGIS Pro by Paul A. Zandbergen](https://www.esri.com/about/newsroom/publications/books/python-scripting-for-arcgis-pro/):** An industry-standard textbook for diving deeper into `arcpy` and automation.
* **[Geo-Python and AutoGIS Courses](https://geo-python-site.readthedocs.io/en/latest/):** An open access course through the University of Helsinki.
* **[Python GIS (pythongis.org)](https://pythongis.org/):** An exceptional, free open-access online textbook that bridges core Python essentials (like Pandas data manipulation and scripting) directly into spatial data science, covering GeoPandas, vector/raster processing, and spatial network analysis.

### General Python & Data Science
* **[Automate the Boring Stuff with Python by Al Sweigart](https://automatetheboringstuff.com/):** A fantastic free online book for picking up practical programming logic, file handling, and web requests.
* **[Spatial Thoughts (spatialthoughts.com)](https://spatialthoughts.com/):** Excellent, practitioner-focused tutorials on QGIS, Google Earth Engine, and Python/Pandas for spatial data analysis.
* **[Pandas Official Documentation & Tutorials](https://pandas.pydata.org/docs/getting_started/index.html):** Essential for mastering data manipulation, grouping, and cleaning.

### Community & Troubleshooting
* **[GIS Stack Exchange (gis.stackexchange.com)](https://gis.stackexchange.com/):** The go-to Q&A community for spatial problems, `arcpy` errors, and Python scripts. 
* **[Esri Community Forums](https://community.esri.com/):** Great for asking platform-specific questions about ArcGIS Pro toolboxes, Arcade expressions, and AGOL integration.